<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/18-computer-vision/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 18: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.transforms import v2


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, torch.Tensor) or isinstance(b, torch.Tensor):
        if not (isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor)):
            return False
        a, b = a.detach().cpu().numpy(), b.detach().cpu().numpy()
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, torch.Tensor):
        body = np.array2string(x.detach().cpu().numpy(), precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"tensor(shape={tuple(x.shape)}, dtype={str(x.dtype).replace('torch.', '')}, {' '.join(body.split())})"
    elif isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, (np.ndarray, torch.Tensor)) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_conv_output_size(f):
    def actual(n, k, p, s, d):
        return F.conv2d(torch.zeros(1, 1, n, n), torch.zeros(1, 1, k, k), padding=p, stride=s, dilation=d).shape[-1]

    cases = [(28, 3, 0, 1, 1), (32, 5, 2, 2, 1), (7, 2, 0, 2, 1), (224, 7, 3, 2, 1), (15, 3, 2, 1, 2), (64, 3, 0, 3, 3)]
    _scenarios("conv_output_size", [
        ("padding 1 keeps the size", lambda: f(28, 3, padding=1), 28, None),
        ("stride 2 halves it", lambda: f(28, 3, padding=1, stride=2), 14, None),
        ("rounds down when the stride doesn't divide evenly", lambda: f(7, 2, stride=2), 3, None),
        ("the same as PyTorch on six layers", lambda: [f(n, k, padding=p, stride=s, dilation=d) for n, k, p, s, d in cases],
         [actual(*c) for c in cases], "with dilation, the kernel covers dilation * (k - 1) + 1 pixels"),
        ("returns an int", lambda: type(f(28, 3)).__name__, "int", "use integer division //"),
        ("a kernel that doesn't fit raises", lambda: f(4, 5), Raises(ValueError), None),
    ])


def check_image_to_tensor(f):
    rng = np.random.default_rng(0)
    colour = rng.integers(0, 256, size=(4, 5, 3), dtype=np.uint8)
    grey = rng.integers(0, 256, size=(6, 7), dtype=np.uint8)
    original = colour.copy()
    expected = torch.tensor(colour, dtype=torch.float32).permute(2, 0, 1) / 255

    def unchanged():
        f(colour)
        return colour

    _scenarios("image_to_tensor", [
        ("colour image shape", lambda: tuple(f(colour).shape), (3, 4, 5), None),
        ("channels in the right place", lambda: f(colour), expected, "use permute, not reshape"),
        ("greyscale gets one channel", lambda: tuple(f(grey).shape), (1, 6, 7), None),
        ("greyscale values", lambda: f(grey)[0], torch.tensor(grey, dtype=torch.float32) / 255, None),
        ("float32", lambda: str(f(colour).dtype), "torch.float32", None),
        ("255 becomes 1.0", lambda: f(np.full((2, 2, 3), 255, dtype=np.uint8)).max().item(), 1.0, None),
        ("the input is not changed", unchanged, original, None),
        ("floats are refused", lambda: f(colour / 255), Raises(ValueError), None),
    ])


def check_conv_parameter_count(f):
    def torch_count(i, o, k, bias=True):
        return sum(p.numel() for p in nn.Conv2d(i, o, k, bias=bias).parameters())

    _scenarios("conv_parameter_count", [
        ("the example", lambda: f(3, 16, 3), 448, "out x in x k x k weights, plus out biases"),
        ("a bigger layer", lambda: f(64, 128, 3), torch_count(64, 128, 3), None),
        ("without bias", lambda: f(64, 128, 3, bias=False), torch_count(64, 128, 3, bias=False), None),
        ("a rectangular kernel", lambda: f(8, 4, (1, 5)), torch_count(8, 4, (1, 5)), "kernel_size can be a pair"),
        ("a 1 x 1 convolution", lambda: f(256, 64, 1, bias=False), torch_count(256, 64, 1, bias=False), None),
    ])


def check_top_k_labels(f):
    names = ["cat", "dog", "fox", "owl", "bee"]
    logits = torch.tensor([0.5, 2.0, -1.0, 2.5, 0.0])
    p = torch.softmax(logits, dim=0)

    def probs(result):
        return [prob for _, prob in result]

    _scenarios("top_k_labels", [
        ("the example", lambda: f(torch.tensor([1.0, 3.0, 2.0]), ["cat", "dog", "fox"], k=2),
         [("dog", torch.softmax(torch.tensor([1.0, 3.0, 2.0]), 0)[1].item()), ("fox", torch.softmax(torch.tensor([1.0, 3.0, 2.0]), 0)[2].item())], None),
        ("names in order, most likely first", lambda: [name for name, _ in f(logits, names, k=3)], ["owl", "dog", "cat"], None),
        ("probabilities use all the classes", lambda: probs(f(logits, names, k=2)), [p[3].item(), p[1].item()],
         "softmax over every class, then pick the top k"),
        ("probabilities are Python floats", lambda: [type(x).__name__ for x in probs(f(logits, names, k=2))], ["float", "float"], "use .tolist() or .item()"),
        ("k = 1", lambda: len(f(logits, names, k=1)), 1, None),
        ("k too large raises", lambda: f(logits, names, k=6), Raises(ValueError), None),
    ])


def check_conv2d_single(f):
    rng = np.random.default_rng(1)
    image = rng.normal(size=(9, 11))
    asym = np.array([[1.0, 2.0, 0.0], [0.0, -1.0, 3.0], [-2.0, 0.5, 1.0]])
    wide = rng.normal(size=(2, 3))

    def torch_conv(img, k, padding=0, stride=1):
        out = F.conv2d(torch.tensor(img).reshape(1, 1, *img.shape), torch.tensor(k).reshape(1, 1, *k.shape),
                       padding=padding, stride=stride)
        return out[0, 0].numpy()

    _scenarios("conv2d_single", [
        ("the lesson's tiny edge example", lambda: f(np.array([[0, 0, 0, 9, 9]] * 4, dtype=float), np.array([[-1.0, 0, 1]] * 3)),
         np.array([[0.0, 27, 27], [0, 27, 27]]), None),
        ("an asymmetric kernel", lambda: f(image, asym), torch_conv(image, asym), "don't flip the kernel"),
        ("with padding", lambda: f(image, asym, padding=1), torch_conv(image, asym, padding=1), None),
        ("with stride 2", lambda: f(image, asym, stride=2), torch_conv(image, asym, stride=2), None),
        ("a rectangular kernel, padding and stride", lambda: f(image, wide, padding=2, stride=3), torch_conv(image, wide, padding=2, stride=3),
         "kernel height and width can differ"),
        ("a 1 x 4 kernel", lambda: f(image, wide[:1, :] @ np.ones((3, 4))), torch_conv(image, wide[:1, :] @ np.ones((3, 4))), None),
        ("returns float64", lambda: str(f(np.ones((4, 4), dtype=np.uint8), np.ones((2, 2))).dtype), "float64", None),
    ])


def check_max_pool2d(f):
    rng = np.random.default_rng(2)
    fm = rng.normal(size=(7, 9))
    ints = rng.integers(0, 100, size=(6, 6))

    def torch_pool(a, size, stride=None):
        return F.max_pool2d(torch.tensor(a, dtype=torch.float64).reshape(1, 1, *a.shape), size, stride)[0, 0].numpy()

    _scenarios("max_pool2d", [
        ("the lesson's example", lambda: f(np.array([[1., 3, 2, 0], [4, 2, 1, 1], [0, 1, 5, 6], [2, 2, 7, 3]])),
         np.array([[4.0, 2], [2, 7]]), None),
        ("an odd size drops the edge", lambda: f(fm), torch_pool(fm, 2), "windows that hang over the edge are dropped"),
        ("size 3", lambda: f(fm, size=3), torch_pool(fm, 3), None),
        ("overlapping windows: size 3, stride 2", lambda: f(fm, size=3, stride=2), torch_pool(fm, 3, 2), "stride defaults to size only when it's None"),
        ("keeps an integer dtype", lambda: str(f(ints).dtype), str(ints.dtype), None),
    ])


def _measured_receptive_field(layers):
    torch.manual_seed(0)
    modules = [nn.Conv2d(1, 1, k, stride=s, bias=False) for k, s in layers]
    for m in modules:
        nn.init.constant_(m.weight, 1.0)
    size = 1
    for k, s in reversed(layers):
        size = (size - 1) * s + k
    size += 4
    x = torch.zeros(1, 1, size, size, requires_grad=True)
    out = x
    for m in modules:
        out = m(out)
    out[0, 0, 0, 0].backward()
    touched = torch.nonzero(x.grad[0, 0].abs().sum(dim=1) > 0).flatten()
    return int(touched.max() - touched.min() + 1)


def check_receptive_field(f):
    rng = np.random.default_rng(3)
    stacks = [[(int(rng.choice([1, 2, 3, 5, 7])), int(rng.choice([1, 1, 2, 3]))) for _ in range(int(rng.integers(2, 6)))] for _ in range(6)]
    _scenarios("receptive_field", [
        ("two 3 x 3 convolutions", lambda: f([(3, 1), (3, 1)]), 5, None),
        ("with a pooling between them", lambda: f([(3, 1), (2, 2), (3, 1)]), 8, "multiply the jump by each stride"),
        ("ResNet's stem", lambda: f([(7, 2), (3, 2)]), 11, None),
        ("the input alone", lambda: f([]), 1, None),
        ("six random stacks, measured with autograd", lambda: [f(s) for s in stacks], [_measured_receptive_field(s) for s in stacks],
         "grow the field by (kernel_size - 1) * jump before updating the jump"),
    ])


def check_pad_and_crop(f):
    g = torch.Generator().manual_seed(4)
    image = torch.rand(3, 6, 8, generator=g)
    positions = [(0, 0), (4, 1), (2, 2), (1, 4), (4, 4)]

    def torchvision_crop(top, left, padding=2):
        return v2.functional.crop(image, top - padding, left - padding, 6, 8)

    _scenarios("pad_and_crop", [
        ("the middle position returns the image", lambda: f(image, 2, 2, 2), image, None),
        ("top = 0, left = 0 moves it down and right", lambda: f(image, 2, 0, 0), torchvision_crop(0, 0), None),
        ("five positions match torchvision", lambda: [f(image, 2, t, l) for t, l in positions],
         [torchvision_crop(t, l) for t, l in positions], "top is a row of the padded image, left a column"),
        ("keeps the shape", lambda: tuple(f(image, 3, 5, 1).shape), (3, 6, 8), None),
        ("a position outside the padding raises", lambda: f(image, 2, 5, 0), Raises(ValueError), None),
    ])


def check_make_transfer_model(f):
    import torchvision

    def build(n_classes, train_last_stage=False):
        torch.manual_seed(0)
        return f(torchvision.models.resnet18(weights=None), n_classes, train_last_stage=train_last_stage)

    def trainable(m):
        return sum(p.numel() for p in m.parameters() if p.requires_grad)

    def trainable_names(m):
        return sorted({name.split(".")[0] for name, p in m.named_parameters() if p.requires_grad})

    _scenarios("make_transfer_model", [
        ("the new last layer", lambda: (build(6).fc.in_features, build(6).fc.out_features), (512, 6), None),
        ("only the new layer is trainable", lambda: trainable(build(6)), 3078, "freeze first, then create the new layer"),
        ("which parts train", lambda: trainable_names(build(6)), ["fc"], None),
        ("with the last stage", lambda: trainable(build(6, train_last_stage=True)), 8396806, None),
        ("which parts train, with the last stage", lambda: trainable_names(build(2, train_last_stage=True)), ["fc", "layer4"], None),
        ("returns the model", lambda: type(build(3)).__name__, "ResNet", None),
        ("works for ResNet-50 too", lambda: f(torchvision.models.resnet50(weights=None), 4).fc.in_features, 2048,
         "read the number of inputs from model.fc.in_features"),
        ("the output has n_classes columns", lambda: tuple(build(3).eval()(torch.zeros(2, 3, 64, 64)).shape), (2, 3), None),
    ])


def check_conv2d_batch(f):
    g = torch.Generator().manual_seed(5)
    x = torch.randn(2, 3, 9, 10, generator=g, dtype=torch.float64)
    w = torch.randn(4, 3, 3, 3, generator=g, dtype=torch.float64)
    b = torch.randn(4, generator=g, dtype=torch.float64)
    w_rect = torch.randn(5, 3, 2, 4, generator=g, dtype=torch.float64)

    def grads(conv):
        xs, ws, bs = (t.clone().requires_grad_(True) for t in (x, w, b))
        (conv(xs, ws, bs, stride=2, padding=1) * torch.arange(4 * 5 * 5, dtype=torch.float64).reshape(4, 5, 5)).sum().backward()
        return [xs.grad, ws.grad, bs.grad]

    def torch_conv(xx, ww, bb=None, stride=1, padding=0):
        return F.conv2d(xx, ww, bb, stride=stride, padding=padding)

    _scenarios("conv2d_batch", [
        ("no padding, stride 1", lambda: f(x, w), F.conv2d(x, w), None),
        ("with bias", lambda: f(x, w, b), F.conv2d(x, w, b), "add the bias to every position of its channel"),
        ("padding 1", lambda: f(x, w, b, padding=1), F.conv2d(x, w, b, padding=1), None),
        ("stride 2 on an odd size", lambda: f(x, w, b, stride=2, padding=1), F.conv2d(x, w, b, stride=2, padding=1),
         "each slice must have exactly H_out rows and W_out columns"),
        ("a rectangular kernel, stride 3", lambda: f(x, w_rect, stride=3, padding=2), F.conv2d(x, w_rect, stride=3, padding=2), None),
        ("gradients match PyTorch's", lambda: grads(f), grads(torch_conv), None),
    ])


def _torchvision_block(c_in, c_out, stride):
    from torchvision.models.resnet import BasicBlock
    torch.manual_seed(0)
    downsample = None
    if stride != 1 or c_in != c_out:
        downsample = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False), nn.BatchNorm2d(c_out))
    block = BasicBlock(c_in, c_out, stride=stride, downsample=downsample)
    with torch.no_grad():
        for m in block.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.weight.uniform_(0.5, 1.5)
                m.bias.uniform_(-0.5, 0.5)
                m.running_mean.uniform_(-0.2, 0.2)
                m.running_var.uniform_(0.5, 1.5)
    return block


def _copy_into(cls, c_in, c_out, stride):
    reference = _torchvision_block(c_in, c_out, stride)
    mine = cls(c_in, c_out, stride)
    state = {k.replace("downsample", "shortcut"): v for k, v in reference.state_dict().items()}
    mine.load_state_dict(state)
    return reference, mine


def check_ResidualBlock(cls):
    x = torch.randn(2, 16, 12, 12, generator=torch.Generator().manual_seed(6))

    def n_params(*args):
        return sum(p.numel() for p in cls(*args).parameters())

    def yours(c_in, c_out, stride, train_mode=False):
        _, mine = _copy_into(cls, c_in, c_out, stride)
        mine.train(train_mode)
        with torch.no_grad():
            return mine(x[:, :c_in])

    def reference(c_in, c_out, stride, train_mode=False):
        block = _torchvision_block(c_in, c_out, stride)
        block.train(train_mode)
        with torch.no_grad():
            return block(x[:, :c_in])

    def shortcut_kind(*args):
        return [type(m).__name__ for m in cls(*args).shortcut]

    _scenarios("ResidualBlock", [
        ("same shape: output shape", lambda: tuple(cls(16, 16)(x).shape), (2, 16, 12, 12), None),
        ("downsampling: output shape", lambda: tuple(cls(16, 32, 2)(x).shape), (2, 32, 6, 6), None),
        ("the identity shortcut has no layers", lambda: shortcut_kind(16, 16), [], "use an empty nn.Sequential()"),
        ("the projection shortcut", lambda: shortcut_kind(16, 32, 2), ["Conv2d", "BatchNorm2d"], None),
        ("same channels with stride 2 needs a projection too", lambda: tuple(cls(16, 16, 2)(x).shape), (2, 16, 6, 6), None),
        ("parameters, same shape", lambda: n_params(16, 16), sum(p.numel() for p in _torchvision_block(16, 16, 1).parameters()), "no bias in the convolutions"),
        ("parameters, downsampling", lambda: n_params(16, 32, 2), sum(p.numel() for p in _torchvision_block(16, 32, 2).parameters()), None),
        ("matches torchvision, same shape", lambda: yours(16, 16, 1), reference(16, 16, 1), "the final ReLU comes after the addition"),
        ("matches torchvision, downsampling", lambda: yours(16, 32, 2), reference(16, 32, 2), "the stride belongs in conv1 and in the shortcut"),
        ("matches torchvision in training mode", lambda: yours(8, 24, 1, True), reference(8, 24, 1, True), None),
    ])


def check_conv2d_backward(f):
    g = torch.Generator().manual_seed(7)

    def case(n, c_in, h, w, c_out, kh, kw, stride, padding):
        x = torch.randn(n, c_in, h, w, generator=g, dtype=torch.float64)
        weight = torch.randn(c_out, c_in, kh, kw, generator=g, dtype=torch.float64)
        out_shape = F.conv2d(x, weight, stride=stride, padding=padding).shape
        grad_out = torch.randn(out_shape, generator=g, dtype=torch.float64)
        xs, ws = x.clone().requires_grad_(True), weight.clone().requires_grad_(True)
        F.conv2d(xs, ws, stride=stride, padding=padding).backward(grad_out)
        return (x, weight, grad_out, stride, padding), (xs.grad, ws.grad)

    simple = case(1, 1, 5, 5, 1, 3, 3, 1, 0)
    channels = case(2, 3, 7, 8, 4, 3, 3, 1, 1)
    strided = case(2, 2, 9, 9, 3, 3, 3, 2, 1)
    gaps = case(1, 2, 8, 8, 2, 2, 2, 3, 0)
    rect = case(2, 3, 6, 9, 2, 2, 4, 2, 2)

    def run(c):
        x, weight, grad_out, stride, padding = c[0]
        return f(x, weight, grad_out, stride=stride, padding=padding)

    _scenarios("conv2d_backward", [
        ("one channel, no padding", lambda: run(simple), simple[1], None),
        ("several channels, padding 1", lambda: run(channels), channels[1], "remove the padding from the input gradient at the end"),
        ("stride 2", lambda: run(strided), strided[1], "add into strided slices, with += where they overlap"),
        ("stride larger than the kernel", lambda: run(gaps), gaps[1], None),
        ("a rectangular kernel", lambda: run(rect), rect[1], None),
        ("shapes", lambda: [tuple(t.shape) for t in run(channels)], [tuple(t.shape) for t in channels[1]], None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `conv_output_size`

In [2]:
def conv_output_size(n: int, kernel_size: int, padding: int = 0, stride: int = 1, dilation: int = 1) -> int:
    covered = dilation * (kernel_size - 1) + 1
    size = (n + 2 * padding - covered) // stride + 1
    if size < 1:
        raise ValueError("the kernel is larger than the padded input")
    return size

In [3]:
check_conv_output_size(conv_output_size)

PASS  all 6 tests passed for conv_output_size


The padded input has `n + 2 * padding` positions. A kernel covering `covered` pixels fits in `n + 2 * padding - covered + 1` positions with stride 1; with stride `s` only every `s`-th of them is used, which gives the `// s + 1`. Integer division rounds down, so when the stride doesn't divide evenly, the last few pixels are simply never visited. That's why the 7 x 7 maps in the lesson's CNN pooled down to 3 x 3, not 4 x 4.

Dilation is a way to see a larger area without more weights or more pooling: a 3 x 3 kernel with dilation 2 covers 5 x 5 pixels using 9 weights. It's common in segmentation networks, which need a wide view but want to keep the image large.

Being able to work out shapes by hand saves a lot of time. The most common error when building a CNN is a final `nn.Linear` whose input size doesn't match the flattened feature maps, and this formula is how you work out the right number (or use `nn.AdaptiveAvgPool2d`, which makes the size fixed whatever the input).

### Exercise 2: `image_to_tensor`

In [4]:
def image_to_tensor(image: np.ndarray) -> torch.Tensor:
    if image.dtype != np.uint8:
        raise ValueError("expected an image of uint8 values")
    if image.ndim == 2:
        image = image[:, :, None]
    return torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255

In [5]:
check_image_to_tensor(image_to_tensor)

PASS  all 8 tests passed for image_to_tensor


`permute(2, 0, 1)` reorders the dimensions: the new first dimension is the old dimension 2 (channels), then the old 0 (height) and 1 (width). It's tempting to use `reshape(C, H, W)` instead, which gives the right shape but the wrong image: reshape keeps the numbers in memory order, so it chops the interleaved red, green and blue values into nonsense. One of the checks uses an image whose three channels are different to catch that.

`torch.tensor(...)` copies the data, so the tensor doesn't share memory with the array. `torch.from_numpy` would share it; that's faster, but the division by 255 creates a new tensor anyway.

The `uint8` check matters more than it looks. An image that has already been converted to floats between 0 and 1 would be divided by 255 again, giving an almost black image, and a network fed such inputs trains badly without any error. torchvision's `v2.ToDtype(torch.float32, scale=True)` does the scaling for you and checks the type in the same way.

### Exercise 3: `conv_parameter_count`

In [6]:
def conv_parameter_count(in_channels: int, out_channels: int, kernel_size, bias: bool = True) -> int:
    if isinstance(kernel_size, int):
        kernel_size = (kernel_size, kernel_size)
    kh, kw = kernel_size
    return out_channels * in_channels * kh * kw + (out_channels if bias else 0)

In [7]:
check_conv_parameter_count(conv_parameter_count)

PASS  all 5 tests passed for conv_parameter_count


The weights have shape `(out_channels, in_channels, kh, kw)`, and there's one bias per output channel. Notice what's missing from the formula: the size of the image. A convolution layer has the same number of parameters for a 28 x 28 image as for a 4000 x 3000 photo, because the same weights are reused at every position. That's the weight sharing from Section 5 of the lesson.

The cost of a layer grows with `in_channels * out_channels`, so doubling the channels of every layer roughly **quadruples** the parameters. That's why CNNs usually double the channels only when pooling halves the image: the amount of computation per layer then stays about the same.

`bias=False` is common when a convolution is followed by batch normalisation, whose shift (`beta`) does the bias's job, as in the lesson's `conv_block`.

### Exercise 4: `top_k_labels`

In [8]:
def top_k_labels(logits: torch.Tensor, class_names, k: int = 3):
    if k > len(class_names):
        raise ValueError("k is larger than the number of classes")
    probabilities = torch.softmax(logits, dim=0)
    top = probabilities.topk(k)
    return [(class_names[i], p) for p, i in zip(top.values.tolist(), top.indices.tolist())]

In [9]:
check_top_k_labels(top_k_labels)

PASS  all 6 tests passed for top_k_labels


`topk` returns both the largest values and their positions, already sorted from largest to smallest. `.tolist()` turns each into a plain Python list, so the probabilities come out as floats and the positions as ints that can index the list of names.

The probabilities must come from the softmax over **all** classes, before choosing the top `k`. Taking the softmax of only the top `k` scores would make them add up to 1 among themselves, so a network that was unsure between 1,000 classes would appear confident, and every probability shown would be too high.

Showing the top few classes rather than only the first is good practice whenever a person reads the result. When the top two are close, as with the lesson's cat (Egyptian cat against tiger cat), the network is telling you that it can't separate them.

## Medium

### Exercise 5: `conv2d_single`

In [10]:
def conv2d_single(image: np.ndarray, kernel: np.ndarray, padding: int = 0, stride: int = 1) -> np.ndarray:
    padded = np.pad(np.asarray(image, dtype=np.float64), padding)
    kh, kw = kernel.shape
    out_h = (padded.shape[0] - kh) // stride + 1
    out_w = (padded.shape[1] - kw) // stride + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            r, c = i * stride, j * stride
            out[i, j] = np.sum(padded[r:r + kh, c:c + kw] * kernel)
    return out

In [11]:
check_conv2d_single(conv2d_single)

PASS  all 7 tests passed for conv2d_single


The loop visits every output position `(i, j)`, finds the matching top-left corner in the padded image (`i * stride`, `j * stride`), and takes the weighted sum of the patch. `np.pad(a, p)` pads every side of every dimension with `p` zeros.

Common slips the checks catch:

- **Flipping the kernel** (true mathematical convolution). For a symmetric kernel such as a blur it makes no difference, so one check uses an asymmetric kernel.
- **Treating the kernel as square.** With a 2 x 3 kernel, height and width must be handled separately.
- **Applying the stride to the patch size** rather than the step, or forgetting it in the output size.

This version is clear but slow: Python runs the inner sum once per output pixel. Exercise 10 does the same arithmetic for whole batches with a handful of tensor operations.

### Exercise 6: `max_pool2d`

In [12]:
def max_pool2d(feature_map: np.ndarray, size: int = 2, stride=None) -> np.ndarray:
    stride = size if stride is None else stride
    out_h = (feature_map.shape[0] - size) // stride + 1
    out_w = (feature_map.shape[1] - size) // stride + 1
    out = np.empty((out_h, out_w), dtype=feature_map.dtype)
    for i in range(out_h):
        for j in range(out_w):
            r, c = i * stride, j * stride
            out[i, j] = feature_map[r:r + size, c:c + size].max()
    return out

In [13]:
check_max_pool2d(max_pool2d)

PASS  all 5 tests passed for max_pool2d


Pooling uses the same output-size formula as a convolution (with no padding), so the structure of the loop is identical to `conv2d_single`; only the operation on each window changes, from a weighted sum to a maximum. Pooling has no weights to learn.

Overlapping pooling (`stride` smaller than `size`) was used in AlexNet; a 3 x 3 window with stride 2 is what ResNet's first pooling layer does. One check uses that combination.

A shortcut for the non-overlapping case is to reshape: crop the map to a multiple of `size`, reshape to `(out_h, size, out_w, size)` and take the maximum over axes 1 and 3. It's much faster, but it only works when `stride == size`.

During backpropagation, the gradient of a max-pooling layer flows only to the position that held the maximum in each window; the other positions get zero. That's part of why pooling makes networks tolerant to small shifts: which pixel "won" can change without changing the output.

### Exercise 7: `receptive_field`

In [14]:
def receptive_field(layers) -> int:
    field, jump = 1, 1
    for kernel_size, stride in layers:
        field += (kernel_size - 1) * jump
        jump *= stride
    return field

In [15]:
check_receptive_field(receptive_field)

PASS  all 5 tests passed for receptive_field


A layer with kernel `k` lets each unit combine `k` neighbouring units of the layer below. Those neighbours are `jump` input pixels apart, so the view grows by `(k - 1) * jump` pixels. A layer with stride `s` then spreads its own units `s` times further apart than the layer below, which is why the jump multiplies.

This is why pooling and strides matter so much. Without them, every 3 x 3 convolution adds only 2 pixels to the receptive field, and you'd need 100 layers to see a 200-pixel object. With a stride of 2 every few layers, the jump doubles each time and the view grows exponentially. ResNet-18's first two layers (a 7 x 7 convolution with stride 2 and a 3 x 3 pooling with stride 2) already give a receptive field of 11 pixels and a jump of 4.

The check verifies your formula on random stacks by measuring the receptive field directly: it builds the layers in PyTorch, backpropagates from one output unit, and counts how many input pixels get a non-zero gradient. That's a handy technique for any network whose structure is too complicated to work out by hand.

### Exercise 8: `pad_and_crop`

In [16]:
def pad_and_crop(image: torch.Tensor, padding: int, top: int, left: int) -> torch.Tensor:
    if not (0 <= top <= 2 * padding and 0 <= left <= 2 * padding):
        raise ValueError("top and left must be between 0 and 2 * padding")
    _, h, w = image.shape
    padded = F.pad(image, (padding, padding, padding, padding))
    return padded[:, top:top + h, left:left + w]

In [17]:
check_pad_and_crop(pad_and_crop)

PASS  all 5 tests passed for pad_and_crop


`F.pad` takes the padding for the **last** dimension first: `(left, right, top, bottom)`. With the same amount everywhere the order doesn't matter, which is convenient.

Why does a smaller `top` move the content **down**? The window starts higher up in the padded image, so it includes more of the zero border above the picture, and the picture itself appears lower in the window. A `top` of 0 shows the full padding at the top and cuts `padding` rows off the bottom of the picture.

The check compares your function with torchvision's own `v2.functional.crop`, which accepts negative positions and fills them with zeros: `crop(image, top - padding, left - padding, H, W)` gives the same result.

For Fashion-MNIST, padding with zeros means padding with black, which matches the background. For photos, padding with black creates artificial black borders, so photo augmentation usually crops and **resizes** instead (`v2.RandomResizedCrop`), or pads by reflecting the edge of the image (`padding_mode="reflect"`).

### Exercise 9: `make_transfer_model`

In [18]:
def make_transfer_model(model: nn.Module, n_classes: int, train_last_stage: bool = False) -> nn.Module:
    for p in model.parameters():
        p.requires_grad = False
    if train_last_stage:
        for p in model.layer4.parameters():
            p.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, n_classes)
    return model

In [19]:
check_make_transfer_model(make_transfer_model)

PASS  all 8 tests passed for make_transfer_model


The order matters slightly: freezing everything first and then creating the new layer means the new layer is trainable automatically, since a freshly created `nn.Linear` has `requires_grad=True`. Creating the new layer first and then freezing "everything" would freeze it too, and the model would have nothing to train. One of the checks looks for exactly that.

Reading `model.fc.in_features` instead of writing 512 makes the function work for other ResNets: ResNet-50's last layer has 2,048 inputs.

Freezing does three things: the frozen weights don't change, PyTorch doesn't compute or store their gradients (saving memory and time), and with very little data, the network has far fewer parameters to overfit with. ResNet-18 with only a new last layer for 6 classes has 3,078 trainable parameters out of over 11 million.

One trap remains when fine-tuning: frozen **batch normalisation** layers still update their running mean and variance in training mode. With small batches of unusual images that can quietly damage the pretrained features. Keeping the frozen part in `eval()` mode, or caching its outputs once as the mini project does, avoids it.

## Hard

### Exercise 10: `conv2d_batch`

In [20]:
def conv2d_batch(x: torch.Tensor, weight: torch.Tensor, bias=None, stride: int = 1, padding: int = 0) -> torch.Tensor:
    n, c_in, h, w = x.shape
    c_out, _, kh, kw = weight.shape
    xp = F.pad(x, (padding, padding, padding, padding))
    out_h = (h + 2 * padding - kh) // stride + 1
    out_w = (w + 2 * padding - kw) // stride + 1
    out = x.new_zeros((n, c_out, out_h, out_w))
    for i in range(kh):
        for j in range(kw):
            patch = xp[:, :, i:i + stride * (out_h - 1) + 1:stride, j:j + stride * (out_w - 1) + 1:stride]
            out = out + torch.einsum("nchw,oc->nohw", patch, weight[:, :, i, j])
    if bias is not None:
        out = out + bias.view(1, -1, 1, 1)
    return out

In [21]:
check_conv2d_batch(conv2d_batch)

PASS  all 6 tests passed for conv2d_batch


Think of a 3 x 3 kernel as nine separate weights per pair of channels. The top-left weight is multiplied by the top-left pixel of every patch; across all output positions, those pixels form a grid that starts at row 0, column 0 of the padded input and steps by `stride`. So for each kernel position `(i, j)` there's one strided slice, and the convolution is the sum of nine "slice times weight matrix" products. The loop runs `kH * kW` times (9 for a 3 x 3 kernel) instead of once per output pixel, and each step is fast tensor arithmetic.

`torch.einsum("nchw,oc->nohw", patch, w)` multiplies each input channel `c` by its weight for output channel `o` and sums over `c`: a matrix product applied at every image and position.

The end of the slice, `i + stride * (out_h - 1) + 1`, makes sure it has exactly `out_h` rows. A plain `i::stride` would usually work but can give one row too many when the stride doesn't divide evenly; one check uses such a case.

Real libraries use the same idea in a different form, called **im2col**: copy every patch into a column of a big matrix (that's what `F.unfold` does), then do one large matrix multiplication, which graphics cards are extremely good at. The check also backpropagates through your function and compares the gradients with PyTorch's, so everything has to be done with tensor operations that autograd can follow: no NumPy and no `.detach()`.

### Exercise 11: `ResidualBlock`

In [22]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        if stride == 1 and in_channels == out_channels:
            self.shortcut = nn.Sequential()
        else:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))

In [23]:
check_ResidualBlock(ResidualBlock)

PASS  all 10 tests passed for ResidualBlock


The addition `out + self.shortcut(x)` needs both tensors to have the same shape. When the block halves the image (stride 2) or changes the number of channels, the input no longer fits, so the shortcut uses a **1 x 1 convolution**: it looks at one pixel at a time and mixes its channels, which changes the number of channels, and with stride 2 it also keeps every other pixel. It's the cheapest layer that makes the shapes match.

Details the checks look at:

- **The stride goes in `conv1`, not `conv2`.** Both give the right output shape, so only the comparison with torchvision's block catches it.
- **The final ReLU comes after the addition.** Applying it to the main path before adding gives different results, and the output could then be negative.
- **No ReLU after `bn2`.** The correction has to be able to be negative, to make some values smaller.
- **The identity shortcut has no parameters.** Using a 1 x 1 convolution everywhere would still work, but adds weights the block doesn't need, and loses the clean "pass the input straight through" path that makes very deep networks trainable.

Stack eight of these blocks, two per stage with 64, 128, 256 and 512 channels, add the 7 x 7 stem and a final linear layer, and you've rebuilt ResNet-18.

### Exercise 12: `conv2d_backward`

In [24]:
def conv2d_backward(x: torch.Tensor, weight: torch.Tensor, grad_out: torch.Tensor, stride: int = 1, padding: int = 0):
    n, c_in, h, w = x.shape
    _, _, kh, kw = weight.shape
    out_h, out_w = grad_out.shape[-2:]
    xp = F.pad(x, (padding, padding, padding, padding))
    grad_xp = torch.zeros_like(xp)
    grad_weight = torch.zeros_like(weight)
    for i in range(kh):
        for j in range(kw):
            rows = slice(i, i + stride * (out_h - 1) + 1, stride)
            cols = slice(j, j + stride * (out_w - 1) + 1, stride)
            grad_weight[:, :, i, j] = torch.einsum("nohw,nchw->oc", grad_out, xp[:, :, rows, cols])
            grad_xp[:, :, rows, cols] += torch.einsum("nohw,oc->nchw", grad_out, weight[:, :, i, j])
    grad_x = grad_xp[:, :, padding:padding + h, padding:padding + w]
    return grad_x, grad_weight

In [25]:
check_conv2d_backward(conv2d_backward)

PASS  all 6 tests passed for conv2d_backward


For each kernel position, the forward pass is a matrix product between the slice of the input and the matrix `weight[:, :, i, j]`. The gradient of a product with respect to each factor is the incoming gradient multiplied by the other factor (Module 9's chain rule, in matrix form):

- **Weights.** `grad_weight[o, c, i, j]` adds up `grad_out[n, o, h, w] * patch[n, c, h, w]` over every image and every position: how much each output changed the loss, times the input pixel that weight multiplied. Every position contributes because the weight was shared across all of them.
- **Input.** Each input pixel was used by several kernel positions (up to 9 for a 3 x 3 kernel), so its gradient is a **sum**, which is why the code uses `+=` into the slice. Pixels in the padding get gradients too, but they aren't real inputs, so they're cut off at the end.

If you look closely, the input gradient is itself a convolution: `grad_out` convolved with the **flipped** kernel, with the channels swapped. That's why some textbooks describe backpropagation through a convolution as a "transposed convolution", and why `nn.ConvTranspose2d`, used to make images larger in segmentation and image-generation networks, is built from the same arithmetic.

With strides, some input pixels are touched by fewer kernel positions, or none at all when the stride is larger than the kernel, and their gradient is correspondingly smaller or zero. One check uses such a case.

## What's next

The mini project in `04_mini_project.ipynb` puts transfer learning to work on real photos: 2,527 pictures of rubbish in six classes, to be sorted for recycling. You'll compare a CNN trained from scratch with a pretrained ResNet-18 used as a feature extractor and then fine-tuned, test the best one once, and look at the mistakes it makes.